In [ ]:
import pandas as pd
import pytz
from spark_config import *

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/15 22:27:21 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [2]:
spark.sql("select t_stamp from sola_ts4 limit 1").show()

Hive Session ID = 32e9d403-3808-4e06-9e1d-b46fb2c59ca1
25/08/15 22:27:27 WARN ConfigurationHelper: Option fs.s3a.connection.establish.timeout is too low (5,000 ms). Setting to 15,000 ms instead


+-------------------+
|            t_stamp|
+-------------------+
|2024-01-01 00:00:00|
+-------------------+



In [4]:
t0 = "2025-01-01 00:00:00+00:00"
t1 = "2025-02-01 00:00:00+00:00"
spark.sql(
    f"select max(t_stamp), min(t_stamp) from sola_ts4 where t_stamp  >=  '{t0}' and t_stamp <  '{t1}'"
).show()

+-------------------+-------------------+
|       max(t_stamp)|       min(t_stamp)|
+-------------------+-------------------+
|2025-01-31 23:55:00|2025-01-01 00:00:00|
+-------------------+-------------------+



In [4]:
df = spark.read.parquet(
    "s3a://project-ciccada/spark-warehouse/bucketed_table/year=2025/month=1"
)

In [7]:
df.createOrReplaceTempView("df")

In [8]:
spark.sql(
    f"select circuit_id, count(distinct t_stamp) as t_stamp_count \
    from df \
    where  t_stamp >= '{t0}' and t_stamp <  '{t1}' and circuit_id = 547781 \
    group by circuit_id \
    order by t_stamp_count desc, circuit_id desc"
).show()

+----------+-------------+
|circuit_id|t_stamp_count|
+----------+-------------+
|    547781|         8921|
+----------+-------------+



In [ ]:
t0 = "2025-01-01 00:00:00+00:00"
t1 = "2025-01-03 00:00:00+00:00"
spark.sql(
    f"select circuit_id, count( t_stamp) as t_stamp_count \
    from sola_ts4 \
    where  t_stamp >= '{t0}' and t_stamp <  '{t1}' and circuit_id = 547781 \
    group by circuit_id \
    order by t_stamp_count desc, circuit_id desc"
).show()

+----------+-------------+
|circuit_id|t_stamp_count|
+----------+-------------+
|    547781|          576|
+----------+-------------+



In [3]:
print(spark.conf.get("spark.sql.warehouse.dir"))

file:/home/hossein/CICCADA/Data_query/spark-warehouse


In [1]:
spark.sql("SHOW DATABASES").show()

NameError: name 'spark' is not defined

In [8]:
spark.sql("SHOW TABLES").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|  default|      sola_ts|      false|
|  default|   sola_sites|      false|
|  default|sola_circuits|      false|
|  default|     sola_ts2|      false|
+---------+-------------+-----------+



In [16]:
spark.sql("describe formatted  SolA_ts").show(50, truncate=False)

+----------------------------+--------------------------------------------------------------+-------+
|col_name                    |data_type                                                     |comment|
+----------------------------+--------------------------------------------------------------+-------+
|circuit_id                  |bigint                                                        |NULL   |
|t_stamp                     |timestamp                                                     |NULL   |
|power                       |double                                                        |NULL   |
|energy                      |double                                                        |NULL   |
|energy_reactive             |double                                                        |NULL   |
|energy_import               |double                                                        |NULL   |
|energy_export               |double                                              

In [6]:
ts_schema = StructType(
    [
        StructField("device_id", LongType()),
        StructField("circuit_id", LongType()),
        StructField("t_stamp", TimestampType()),
        StructField("power", DoubleType()),
        StructField("energy", DoubleType()),
        StructField("energy_reactive", DoubleType()),
        StructField("energy_import", DoubleType()),
        StructField("energy_export", DoubleType()),
        StructField("energy_reactive_import", DoubleType()),
        StructField("energy_reactive_export", DoubleType()),
        StructField("power_factor", DoubleType()),
        StructField("voltage", DoubleType()),
        StructField("current", DoubleType()),
    ]
)

In [10]:
t0 = "2024-01-02 00:00:00+10:00"
t1 = "2024-01-03 00:00:00+10:00"

In [11]:
df = spark.sql(f"""
    select *
    from SolA_ts2 
    where is_pv = True and month=1 and year=2024 and circuit_id = 547781 
                            and t_stamp >= TIMESTAMP '{t0}' and t_stamp < TIMESTAMP '{t1}'
    order by t_stamp desc
""").toPandas()

25/08/06 14:33:47 WARN ConfigurationHelper: Option fs.s3a.connection.establish.timeout is too low (5,000 ms). Setting to 15,000 ms instead


In [15]:
df

,circuit_id,t_stamp,energy,energy_reactive,energy_import,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv
0,547781,2024-01-02 13:55:00,901.8567,48.3317,75.1547,0.0,0.0,0.707427,234.00,4.5685,2024,1,True
1,547781,2024-01-02 13:50:00,900.0533,48.2089,75.0044,0.0,0.0,0.707652,233.75,4.5715,2024,1,True
2,547781,2024-01-02 13:45:00,896.1600,48.0025,74.6800,0.0,0.0,0.707634,233.15,4.5585,2024,1,True
3,547781,2024-01-02 13:40:00,894.4367,48.0022,74.5364,0.0,0.0,0.706839,232.60,4.5705,2024,1,True
4,547781,2024-01-02 13:35:00,894.8200,47.9983,74.5683,0.0,0.0,0.707050,233.00,4.5545,2024,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,547781,2024-01-01 14:20:00,932.9167,50.1625,77.7431,0.0,0.0,0.706052,238.55,4.6000,2024,1,True
282,547781,2024-01-01 14:15:00,931.0767,50.0208,77.5897,0.0,0.0,0.706406,238.35,4.6195,2024,1,True
283,547781,2024-01-01 14:10:00,929.4500,49.9436,77.4542,0.0,0.0,0.706321,238.15,4.6300,2024,1,True
284,547781,2024-01-01 14:05:00,930.3133,49.9978,77.5261,0.0,0.0,0.706256,238.10,4.6080,2024,1,True


In [13]:
df1 = (
    spark.read.option("recursiveFileLookup", "true")
    .schema(ts_schema)
    .parquet(
        f"s3a://project-ciccada/spark-warehouse/bucketed_table2/year=2024/month=1/",
        escape='"',
        multiLine=True,
        quote='"',
        encoding="UTF-8",
    )
    .filter(col("circuit_id") == 547781)
    .toPandas()
)

In [12]:
df[df.duplicated()].shape

(0, 13)

In [14]:
df1["t_stamp"] = (
    df1["t_stamp"].dt.tz_localize("UTC").dt.tz_convert(pytz.FixedOffset(600))
)
df2 = df1.query(f"t_stamp >= '{t0}' and t_stamp < '{t1}'")
df2[df2.duplicated()].shape

(0, 13)

In [ ]:
df1 = (
    spark.read.schema(ts_schema)
    .parquet(
        f"s3a://project-ciccada/spark-warehouse/bucketed_table2/year=2024/month=1/",
        escape='"',
        multiLine=True,
        quote='"',
        encoding="UTF-8",
    )
    .filter(col("circuit_id") == 547781)
    .toPandas()
)

In [22]:
spark.sql(f"""SHOW CREATE TABLE SolA_ts;""").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|createtab_stmt                                                                                                                                                                                                                                                                                                                                                                             

In [ ]:
df = spark.sql(
    f"SELECT circuit_id, t_stamp, power/1000 as power, voltage from SolA_ts \
    where is_pv = True and month=1 and year=2024 \
       and t_stamp >= '{t0}' and t_stamp < '{t1}'"
).toPandas()

Hive Session ID = 8952d4a7-40a3-4663-86f5-e98573d6d2f4
25/08/06 09:06:55 WARN ConfigurationHelper: Option fs.s3a.connection.establish.timeout is too low (5,000 ms). Setting to 15,000 ms instead


In [4]:
df = spark.sql(f"""
    select circuit_id, count(t_stamp) as t_stamp_count
    from SolA_ts 
    where is_pv = True and month=1 and year=2024
    group by circuit_id
    order by t_stamp_count desc, circuit_id desc
""").toPandas()

In [ ]:
spark.sql(
    f"SELECT circuit_id, t_stamp, power/1000 as power, voltage from SolA_ts \
    where is_pv = True and month=1 and year=2024 \
       and t_stamp > '{t0}' and t_stamp < '{t1}' and circuit_id=6066"
).show(10, truncate=False)

In [ ]:
df["t_stamp"] = pd.to_datetime(df["t_stamp"], utc=True)
df["t_stamp"] = df["t_stamp"].dt.tz_convert(pytz.FixedOffset(600))  # Convert to UTC+10

In [ ]:
df["t_stamp"].min(), df["t_stamp"].max()